# Our Approach

## Training one fixed baseline MLP

This notebook makes the first modelling choice for the solar PV mini-project. We will train a small multilayer perceptron using the inputs from `01_problem_setup`, inspect whether training made progress, and check the trained baseline against the project criterion.

The baseline is a reference point, not the final claim. Keep the configuration fixed in this notebook so that everyone starts the investigation from the same trained model.


In [ ]:
import sys
print(sys.executable)

In [ ]:
# Environment setup. The notebook is designed to run locally and in Colab.
import importlib
import importlib.util
import os
import subprocess
import sys
import tempfile
from pathlib import Path

os.environ.setdefault(
    "MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "nextgen2026-matplotlib")
)

REPO_URL = "https://github.com/nextgenerationgraduatesprogram/nextgen2026-mlai-workshops.git"
REPO_BRANCH = "workshop3"
PACKAGE_NAME = "nextgen2026_mlai_workshops"

if "google.colab" in sys.modules:
    repo_dir = Path("/content/nextgen2026-mlai-workshops")
    if not repo_dir.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "--branch",
                REPO_BRANCH,
                REPO_URL,
                str(repo_dir),
            ],
            check=True,
        )
    else:
        subprocess.run(["git", "-C", str(repo_dir), "fetch", "--depth", "1", "origin", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", str(repo_dir), "checkout", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", str(repo_dir), "pull", "--ff-only", "origin", REPO_BRANCH], check=True)

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(repo_dir)], check=True)
    missing_packages = [
        package_name
        for package_name, module_name in (("torch", "torch"),)
        if importlib.util.find_spec(module_name) is None
    ]
    if missing_packages:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing_packages], check=True)
    sys.path.insert(0, str(repo_dir / "src"))
else:
    repo_dir = None
    for possible_root in (Path.cwd(), Path.cwd().parent):
        possible_src = possible_root / "src"
        if (possible_src / PACKAGE_NAME).exists():
            repo_dir = possible_root
            sys.path.insert(0, str(possible_src))
            break

for module_name in list(sys.modules):
    if module_name == PACKAGE_NAME or module_name.startswith(f"{PACKAGE_NAME}."):
        del sys.modules[module_name]

workshop_pkg = importlib.import_module(PACKAGE_NAME)
package_version = getattr(workshop_pkg, "__version__", "unknown")
print(
    f"Workshop 3 environment ready. Package: {PACKAGE_NAME} {package_version}. "
    f"Repository: {repo_dir or 'installed environment'}"
)


<br>

## Learning Outcomes

By the end of this notebook, you should be able to:

1. identify the data, model, loss, and optimizer choices in the fixed baseline;
2. build in-memory PyTorch `Dataset` and `DataLoader` objects for the PV splits;
3. read the `SolarPVMLP` interface used for the baseline model;
4. explain what MSE and plain minibatch SGD are doing operationally;
5. interpret training history and training curves;
6. decide whether the fixed baseline meets the project success criterion.

The central question is:

> Does a reasonable first MLP meet the project criterion, or do we need more evidence about where it works and fails?


<br>

## Setup

This cell loads the same input project bundle introduced in `01_problem_setup`. The split names are project infrastructure: train is used to update parameters, validation is used to monitor the fixed run and apply the criterion, and test is checked once after the baseline configuration is fixed.

The model only receives the input vector $x=(I,T_a,\alpha)$, where $I$ is irradiance, $T_a$ is ambient temperature, and $\alpha$ is tilt angle.

$$
x=(I,T_a,\alpha)\in X \subset\mathbb{R}^3.
$$


In [ ]:
# Load helpers and prepare the fixed baseline bundle and configuration.
from nextgen2026_mlai_workshops import solar_pv as pv
import matplotlib.pyplot as plt
import numpy as np
import torch

bundle = pv.make_workshop3_bundle("baseline", seed=7)
input_columns = ["irradiance", "ambient_temperature", "tilt_angle"]
target_column = bundle["target"]
criterion = bundle["criterion"]
baseline_config = pv.baseline_config()
show_advanced = False

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()} (the workshop helpers use CPU tensors)")
print(f"Inputs: {', '.join(input_columns)}")
print(f"Target: {target_column}")


<br>

## 1. Restate The Modelling Formulation

We now choose a first model for the function defined in `01_problem_setup`. The baseline maps three inputs to normalized PV power:

$$
h_\theta: X \to [0,1],
\qquad
x=(I,T_a,\alpha),
\qquad
\widehat{P}=h_\theta(x).
$$

A small MLP is a reasonable first model because it can represent nonlinear multi-input relationships without requiring hand-written feature rules. The important constraint for this notebook is that the configuration stays fixed before we evaluate it.


In [ ]:
# Recreate the key operating range from the shared criterion object.
criterion_range = (
    f"irradiance >= {criterion.key_irradiance_min:.0f} W/m2; "
    f"ambient_temperature >= {criterion.key_ambient_min:.0f} C; "
    f"{criterion.key_tilt_min:.0f} <= tilt_angle <= {criterion.key_tilt_max:.0f} degrees"
)

pv.print_table(
    ["Item", "Value"],
    [
        ["prediction task", "operating conditions -> normalized power"],
        ["inputs", ", ".join(input_columns)],
        ["target", target_column],
        ["overall validation requirement", f"RMSE <= {criterion.overall_rmse:.3f}"],
        ["key operating-range requirement", f"RMSE <= {criterion.key_range_rmse:.3f}"],
        ["key operating range", criterion_range],
    ],
)


<br>

## 2. Fix The Baseline Configuration

The baseline configuration is set once. These values make the run reproducible and give the class a shared reference model to inspect.

The choices are intentionally plain: standard scaling fitted on train inputs only, a two-hidden-layer ReLU MLP, MSE loss, and SGD without momentum.


In [ ]:
# Print the fixed baseline choices for reproducibility.
config_rows = [
    ["input columns", ", ".join(input_columns)],
    ["split sizes", f"{len(bundle['train'][target_column])} train, {len(bundle['validation'][target_column])} validation, {len(bundle['test'][target_column])} test"],
    ["scaling", baseline_config["scaling"]],
    ["model", "SolarPVMLP"],
    ["hidden width", baseline_config["hidden_width"]],
    ["depth", f"{baseline_config['depth']} hidden layers"],
    ["activation", baseline_config["activation"]],
    ["initialization", baseline_config["initialization"]],
    ["loss", baseline_config["loss"]],
    ["optimizer", "SGD, no momentum"],
    ["learning rate", baseline_config["learning_rate"]],
    ["batch size", baseline_config["batch_size"]],
    ["epochs", baseline_config["epochs"]],
    ["seed", baseline_config["seed"]],
]

pv.print_table(["Component", "Baseline choice"], config_rows)


<br>

## 3. Prepare Tensors And Loaders

PyTorch training works with tensors and minibatches. The cell below keeps the data in memory, fits a scaler on the train inputs, transforms each split with that train-only scaler, and wraps the arrays in `Dataset` and `DataLoader` objects.

Look for three practical details: the model receives three features, targets are column vectors, and a minibatch is smaller than the full training set.


In [ ]:
# Convert arrays to tensors and inspect a representative minibatch.
raw_features = {
    split: pv.feature_matrix(bundle[split], input_columns)
    for split in ("train", "validation", "test")
}
targets = {
    split: pv.target_vector(bundle[split])
    for split in ("train", "validation", "test")
}

scaler = pv.fit_scaler(raw_features["train"], baseline_config["scaling"])
scaled_features = {
    split: pv.transform_with_scaler(raw_features[split], scaler)
    for split in ("train", "validation", "test")
}

datasets = {
    split: pv.PVDataset(scaled_features[split], targets[split])
    for split in ("train", "validation", "test")
}

loader_generator = torch.Generator().manual_seed(baseline_config["seed"] + 123)
loaders = {
    "train": torch.utils.data.DataLoader(
        datasets["train"],
        batch_size=baseline_config["batch_size"],
        shuffle=True,
        generator=loader_generator,
    ),
    "validation": torch.utils.data.DataLoader(datasets["validation"], batch_size=baseline_config["batch_size"], shuffle=False),
    "test": torch.utils.data.DataLoader(datasets["test"], batch_size=baseline_config["batch_size"], shuffle=False),
}

first_x, first_y = next(iter(loaders["train"]))

pv.print_table(
    ["Split", "Examples", "X tensor shape", "y tensor shape", "Loader shuffle"],
    [
        [split, len(datasets[split]), tuple(datasets[split].X.shape), tuple(datasets[split].y.shape), split == "train"]
        for split in ("train", "validation", "test")
    ],
)
print(f"One training minibatch: X {tuple(first_x.shape)}, y {tuple(first_y.shape)}")

pv.print_table(
    ["Feature", "Train mean used for scaling", "Train std used for scaling"],
    [
        [feature, float(center), float(scale)]
        for feature, center, scale in zip(
            input_columns,
            scaler["center"].ravel(),
            scaler["scale"].ravel(),
        )
    ],
)


<br>

## 4. Inspect The MLP Interface

The `SolarPVMLP` class defines the function family available to the baseline. The public interface names the choices that matter for this run: input dimension, width, depth, activation, and initialization.

A parameter count is useful for scale, but it does not prove the model is suitable. Suitability comes from evaluation against the criterion.


In [ ]:
# Instantiate the MLP once to inspect its interface and parameter count.
model_preview = pv.SolarPVMLP(
    input_dim=len(input_columns),
    hidden_width=baseline_config["hidden_width"],
    depth=baseline_config["depth"],
    activation=baseline_config["activation"],
    initialization=baseline_config["initialization"],
)
parameter_count = sum(parameter.numel() for parameter in model_preview.parameters())

print(model_preview)
pv.print_table(
    ["Interface item", "Baseline value"],
    [
        ["class", "pv.SolarPVMLP"],
        ["input_dim", len(input_columns)],
        ["hidden_width", baseline_config["hidden_width"]],
        ["depth", baseline_config["depth"]],
        ["activation", baseline_config["activation"]],
        ["initialization", baseline_config["initialization"]],
        ["trainable parameters", parameter_count],
    ],
)


<br>

## 5. Define The Loss

The loss tells training what kinds of mistakes matter. The baseline uses mean squared error:

$$
\mathcal{L}_{\mathrm{MSE}}(\theta)
=
\frac{1}{n}\sum_{i=1}^{n}
\left(h_\theta(x_i)-P_i\right)^2.
$$

MSE penalizes larger residuals more strongly than smaller residuals. That is a concrete modelling choice for this baseline.


In [ ]:
# Evaluate example residuals to show how the configured loss behaves.
mse_loss = pv.loss_function(baseline_config["loss"])
example_residuals = torch.tensor([0.02, -0.05, 0.20], dtype=torch.float32)
example_penalties = example_residuals**2
example_mse = mse_loss(example_residuals.reshape(-1, 1), torch.zeros(3, 1))

print(mse_loss)
pv.print_table(
    ["Residual", "Squared penalty"],
    [[float(residual), float(penalty)] for residual, penalty in zip(example_residuals, example_penalties)],
)
print(f"Mean squared error for these residuals: {float(example_mse):.4f}")


<br>

## 6. Define Plain SGD

The optimizer specifies how parameters are updated. Plain minibatch SGD estimates a gradient on each minibatch and steps in the negative-gradient direction.

For this baseline, momentum is zero. That keeps the training recipe easy to read and easy to reproduce.


In [ ]:
# Create an optimizer preview so the update rule is explicit before training.
optimizer_preview = torch.optim.SGD(
    model_preview.parameters(),
    lr=baseline_config["learning_rate"],
    momentum=0.0,
    weight_decay=0.0,
)

pv.print_table(
    ["Optimizer setting", "Value"],
    [
        ["optimizer", optimizer_preview.__class__.__name__],
        ["momentum", optimizer_preview.param_groups[0]["momentum"]],
        ["weight decay", optimizer_preview.param_groups[0]["weight_decay"]],
        ["learning rate", optimizer_preview.param_groups[0]["lr"]],
        ["batch size", baseline_config["batch_size"]],
        ["epochs", baseline_config["epochs"]],
        ["seed", baseline_config["seed"]],
    ],
)


<br>

## 7. Train The Baseline

Training combines the data loaders, model class, loss, optimizer, and fixed configuration into one selected model. The helper recreates the same setup from `baseline_config` so later notebooks can rerun the baseline with one reproducible call. It records a history row for every epoch and restores the parameters from the best validation epoch.

Do not change the configuration in this notebook. The purpose is to produce one shared baseline run.


In [ ]:
# Train the fixed baseline model and keep the recorded run object.
baseline_run = pv.train_with_config(bundle, baseline_config, name="Baseline MLP")
history = baseline_run["history"]
best_epoch = baseline_run["best_validation_epoch"]
best_row = history[best_epoch - 1]
final_row = history[-1]
first_row = history[0]

pv.print_table(
    ["Checkpoint", "Epoch", "Train MSE", "Validation MSE", "Train RMSE", "Validation RMSE"],
    [
        ["first", int(first_row["epoch"]), first_row["train_loss"], first_row["validation_loss"], first_row["train_RMSE"], first_row["validation_RMSE"]],
        ["best validation", int(best_row["epoch"]), best_row["train_loss"], best_row["validation_loss"], best_row["train_RMSE"], best_row["validation_RMSE"]],
        ["final", int(final_row["epoch"]), final_row["train_loss"], final_row["validation_loss"], final_row["train_RMSE"], final_row["validation_RMSE"]],
    ],
)
print(f"History rows recorded: {len(history)}")
print(f"Best validation epoch selected for reporting: {best_epoch}")


<br>

## 8. Plot Training Curves

Training curves show whether the optimization process made progress. A useful baseline should reduce loss and produce a sensible validation trace.

The curves do not explain every error pattern. They only tell us whether this fixed run trained in a stable, interpretable way.


In [ ]:
# Plot train and validation loss curves from the recorded run.
pv.plot_training_curves(baseline_run)
plt.show()


<br>

## 9. Evaluate The Baseline On Validation Evidence

Now compare the trained baseline with the project criterion. The report uses validation evidence for the criterion and includes simple input slices so that we do not rely only on one average score.

Look for two things: whether the combined criterion passes, and whether the slice table suggests uneven behaviour across operating ranges.


In [ ]:
# Evaluate the trained run with the shared report.
validation_report = pv.evaluate_model_report(
    baseline_run,
    bundle,
    include_test=False,
    show_advanced=show_advanced,
)
pv.print_report(validation_report)

pv.plot_observed_vs_predicted(baseline_run, bundle, split="validation")
plt.show()


<br>

## 10. Criterion Check

The baseline is acceptable only if it meets both validation requirements. A model can pass one row and still fail the combined criterion.

This is the decision point for the fixed baseline: record the result, but do not revise the configuration here.


In [ ]:
# Print the validation criterion rows that decide baseline acceptability.
criterion_result = validation_report["criterion"]
overall_pass = criterion_result["overall_validation_rmse"] <= criterion_result["overall_threshold"]
key_range_pass = criterion_result["key_range_validation_rmse"] <= criterion_result["key_range_threshold"]
combined_pass = criterion_result["passes"]

pv.print_table(
    ["Check", "Validation value", "Required", "Pass"],
    [
        [
            "Overall RMSE",
            criterion_result["overall_validation_rmse"],
            f"<= {criterion_result['overall_threshold']:.3f}",
            overall_pass,
        ],
        [
            "Key operating-range RMSE",
            criterion_result["key_range_validation_rmse"],
            f"<= {criterion_result['key_range_threshold']:.3f}",
            key_range_pass,
        ],
        ["Combined criterion", "", "both rows must pass", combined_pass],
    ],
)

if combined_pass:
    print("Criterion status: the fixed baseline meets the validation criterion.")
else:
    print("Criterion status: the fixed baseline does not meet the validation criterion.")


<br>

## 11. Final Baseline Test Check

The baseline configuration is now fixed, so we can make one final held-out test check for the baseline record. This is not a place to choose a different model; it is a report of how the fixed baseline behaves on the final split.


In [ ]:
# Run the final held-out check after the choice is fixed.
final_baseline_report = pv.final_check(baseline_run, bundle, show_advanced=show_advanced)
test_row = next(row for row in final_baseline_report["metric_rows"] if row[0] == "test")
validation_row = next(row for row in final_baseline_report["metric_rows"] if row[0] == "validation")

pv.print_table(
    ["Final baseline check", "Validation", "Test"],
    [
        ["RMSE", validation_row[2], test_row[2]],
        ["MAE", validation_row[3], test_row[3]],
        ["Key operating-range RMSE", validation_row[4], test_row[4]],
    ],
)


<br>

## 12. Handoff

We now have a fixed, reasonable baseline MLP and a criterion check. Training made progress, but the validation criterion tells us the baseline is not yet acceptable for the full project claim.

The next notebook uses this fixed run as the shared evaluation starting point and asks what evidence we need before deciding what to change.
